# RMDS Data Science Challenge Hackathon — Dataset C (COVID-19)

Colab-ready end-to-end pipeline covering:
1. **Setup & Data Loading** (`DOCs/dataset_C_training.csv`, `DOCs/dataset_C_testing.csv`)
2. **Exploratory Data Analysis & Cleaning** (whitespace fix, undocumented `0` opinion codes, missingness signals)
3. **Domain Feature Engineering** (doctor-opinion interactions, attitude indices, vulnerability indicators)
4. **Hospitalisation Risk & Cost Matrix** ($1,500 vaccine cost, $41,000 hospitalisation cost, Bayes-optimal threshold $\tau^*(x_i)$)
5. **5-Fold Stratified CV Modeling** (LogisticRegression, RandomForest, HistGradientBoosting, LightGBM, XGBoost, CatBoost, Calibrated Stack)
6. **Explainable AI (SHAP)** & **Top 5 Submission Export**

In [ ]:
# Colab bootstrap: clone repo if running standalone in Google Colab
import os, sys
from pathlib import Path

if not (Path.cwd() / 'src').exists() and not (Path.cwd().parent / 'src').exists():
    !git clone -q https://github.com/Saikumarmuddada/RMDS.git
    os.chdir('RMDS')

# Uncomment in Google Colab if packages are missing:
# !pip install -q lightgbm xgboost catboost shap seaborn

GROUP_ID = 0  # <-- Set your Hackathon Group Number X here
print('Working directory:', Path.cwd())

In [ ]:
# Ensure repo root is on sys.path (whether launched from repo root or notebooks/)
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config import DATASET_C_CONFIG, FIGURES_DIR, OUTPUT_DIR, SUBMISSIONS_DIR
from src.data_prep import load_raw_datasets, engineer_features
from src.eda import run_eda
from src.modeling import train_and_evaluate_all_models, export_top5_submissions
from src.explainability import run_explainability_and_plots

raw_train, raw_test, example_sub = load_raw_datasets(DATASET_C_CONFIG)
feat_train = engineer_features(raw_train, DATASET_C_CONFIG)
feat_test = engineer_features(raw_test, DATASET_C_CONFIG)
pos_rate = feat_train[DATASET_C_CONFIG.target_col].mean()
print(f'Train: {feat_train.shape}, Test: {feat_test.shape}, Positive rate: {pos_rate:.2%}')

In [ ]:
# Step 1: Exploratory Data Analysis & Figures (01-03)
eda_summary = run_eda(raw_train, raw_test, feat_train, DATASET_C_CONFIG, OUTPUT_DIR, FIGURES_DIR)
eda_summary['top_target_correlations'][:8]

In [ ]:
# Step 2: Train 6 Diverse Models + Calibrated Stacking Ensemble (5-fold Stratified CV)
results, comp_df, folds, artifacts = train_and_evaluate_all_models(
    feat_train, feat_test, DATASET_C_CONFIG, OUTPUT_DIR
)
comp_df[['model', 'roc_auc', 'log_loss', 'cost_global_0.5', 'cost_bayes_exact', 'captured_oracle_pct']]

In [ ]:
# Step 3: Explainable AI (SHAP) & Cost-Comparison Visuals (04-07)
explain_summary = run_explainability_and_plots(
    results, comp_df, feat_train, feat_test, artifacts, DATASET_C_CONFIG, OUTPUT_DIR, FIGURES_DIR
)
explain_summary['top_shap_features'][:10]

In [ ]:
# Step 4: Export Top 5 Ranked Canvas Submission Files
manifest_df = export_top5_submissions(
    results, feat_train, feat_test, example_sub,
    group_id=GROUP_ID, cfg=DATASET_C_CONFIG, submissions_dir=SUBMISSIONS_DIR,
)
manifest_df